In [ ]:
!pip install ultralytics stable-baselines3 gymnasium

In [ ]:
import numpy as np
import cv2
import time
from collections import deque
from ultralytics import YOLO
from stable_baselines3 import DQN

In [ ]:
total_count = {
    "N": set(),
    "E": set(),
    "S": set(),
    "W": set()
}

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class TrafficEnvironment(gym.Env):

    def __init__(self):
        super().__init__()

        self.observation_space = spaces.Box(low=0, high=50, shape=(4,), dtype=np.float32)
        self.action_space = spaces.Discrete(4)

        self.max_cars = 50
        self.previous_action = 0

    def reset(self, seed=None, options=None):
        self.state = np.random.randint(5, 25, size=4).astype(np.float32)
        self.previous_action = 0
        return self.state, {}

    def step(self, action):

        # تقليل الزحمة في الاتجاه المختار
        self.state[action] = max(0, self.state[action] - np.random.randint(4, 8))

        # زيادة عشوائية في باقي الاتجاهات
        for i in range(4):
            if i != action:
                self.state[i] += np.random.randint(1, 4)

        self.state = np.clip(self.state, 0, self.max_cars)

        # reward محسّن
        reward = -(np.sum(self.state) + np.max(self.state) * 2)

        # penalty لتغيير الاتجاه بسرعة
        if action != self.previous_action:
            reward -= 3

        self.previous_action = action

        return self.state, reward, False, False, {}

In [ ]:
env = TrafficEnvironment()

model = DQN(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.0005,
    buffer_size=10000,
    batch_size=32
)

model.learn(total_timesteps=30000)
model.save("smart_traffic_model")

In [ ]:
vehicle_detector = YOLO("yolov8n.pt")
traffic_agent = DQN.load("smart_traffic_model")

In [ ]:
import os
print(os.listdir())

In [ ]:
video_path = "/content/WhatsApp Video 2026-05-05 at 1.47.15 PM.mp4"
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('final_output.avi', fourcc, fps, (width, height))

In [ ]:
def assign_lane(cx, cy, w, h):

    if cy < h * 0.4:
        return "N"
    elif cy > h * 0.6:
        return "S"
    elif cx > w * 0.6:
        return "E"
    else:
        return "W"


def calculate_density(lane_ids):
    return [
        len(lane_ids["N"]),
        len(lane_ids["E"]),
        len(lane_ids["S"]),
        len(lane_ids["W"])
    ]


def draw_traffic_signal(frame, action):

    colors = [(0,0,255)] * 4
    colors[action] = (0,255,0)

    labels = ["N", "E", "S", "W"]

    for i in range(4):
        x = 60 + i*80
        cv2.circle(frame, (x, 250), 20, colors[i], -1)
        cv2.putText(frame, labels[i], (x-10, 280),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    return frame

In [ ]:
history = deque(maxlen=5)

def smooth_state(state):
    history.append(state)
    return np.mean(history, axis=0).astype(int)

In [ ]:
def assign_lane(cx, cy, w, h):

    if cy < h * 0.4:
        return "N"
    elif cy > h * 0.6:
        return "S"
    elif cx > w * 0.6:
        return "E"
    else:
        return "W"

In [ ]:
from google.colab.patches import cv2_imshow
from collections import deque
import numpy as np
import time
import cv2

roads = ["North", "East", "South", "West"]

last_switch_time = time.time()
current_green = 0

# نظام التوقيت
min_green = 3
max_green = 12
green_duration = 5

previous_action = 0

# =========================
# Reward variables
# =========================
current_reward = 0
reward_history = deque(maxlen=20)
prev_action_for_reward = 0

# =========================
# NEW: Waiting Time
# =========================
waiting_time = {"N": 0, "E": 0, "S": 0, "W": 0}

frame_count = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # =====================================
    # DETECTION + TRACKING
    # =====================================
    results = vehicle_detector.track(frame, persist=True, classes=[2])

    lane_ids = {"N": set(), "E": set(), "S": set(), "W": set()}

    if results[0].boxes is not None and results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        h, w, _ = frame.shape

        for box, car_id in zip(boxes, ids):

            x1, y1, x2, y2 = map(int, box)

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            lane = assign_lane(cx, cy, w, h)

            lane_ids[lane].add(int(car_id))
            total_count[lane].add(int(car_id))

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

            cv2.putText(frame,
                        f"ID:{int(car_id)} {lane}",
                        (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (0,255,0),
                        2)

    # =====================================
    # حساب الزحمة + smoothing
    # =====================================
    traffic_state = calculate_density(lane_ids)
    traffic_state = smooth_state(traffic_state)

    # =====================================
    # RL DECISION + SMART TIMING
    # =====================================
    if time.time() - last_switch_time > green_duration:

        action, _ = traffic_agent.predict(traffic_state)
        current_green = int(action)

        green_duration = min(
            max_green,
            max(min_green, traffic_state[current_green])
        )

        if current_green == previous_action:
            green_duration += 2
        else:
            green_duration = min_green

        previous_action = current_green
        last_switch_time = time.time()

    action = current_green

    # =====================================
    # NEW: Waiting Time Update
    # =====================================
    lanes = ["N", "E", "S", "W"]

    for i, lane in enumerate(lanes):
        if i == action:
            waiting_time[lane] = max(0, waiting_time[lane] - 2)
        else:
            waiting_time[lane] += traffic_state[i]

    # =====================================
    # REWARD (updated with waiting time)
    # =====================================
    current_reward = -(
        np.sum(traffic_state)
        + np.max(traffic_state) * 2
        + sum(waiting_time.values()) * 0.1
    )

    if action != prev_action_for_reward:
        current_reward -= 3

    prev_action_for_reward = action

    reward_history.append(current_reward)
    avg_reward = np.mean(reward_history)

    # =====================================
    # DISPLAY INFO
    # =====================================
    cv2.putText(frame, f"N: {traffic_state[0]}", (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    cv2.putText(frame, f"E: {traffic_state[1]}", (20,80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    cv2.putText(frame, f"S: {traffic_state[2]}", (20,120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    cv2.putText(frame, f"W: {traffic_state[3]}", (20,160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    # =====================================
    # TOTAL COUNT
    # =====================================
    cv2.putText(frame, f"Total N: {len(total_count['N'])}", (250,40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

    cv2.putText(frame, f"Total E: {len(total_count['E'])}", (250,80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

    cv2.putText(frame, f"Total S: {len(total_count['S'])}", (250,120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

    cv2.putText(frame, f"Total W: {len(total_count['W'])}", (250,160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

    # =====================================
    # WAITING TIME DISPLAY (NEW)
    # =====================================
    cv2.putText(frame, f"WT N: {waiting_time['N']}", (450,40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

    cv2.putText(frame, f"WT E: {waiting_time['E']}", (450,80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

    cv2.putText(frame, f"WT S: {waiting_time['S']}", (450,120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

    cv2.putText(frame, f"WT W: {waiting_time['W']}", (450,160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

    # =====================================
    # REWARD DISPLAY
    # =====================================
    cv2.putText(frame, f"Reward: {int(current_reward)}", (20, 320),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 3)

    cv2.putText(frame, f"Avg Reward: {int(avg_reward)}", (20, 360),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

    # =====================================
    # GREEN SIGNAL
    # =====================================
    cv2.putText(frame, f"GREEN: {roads[action]}", (20,210),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 3)

    frame = draw_traffic_signal(frame, action)

    out.write(frame)

    if frame_count % 10 == 0:
        cv2_imshow(frame)

cap.release()
out.release()

In [ ]:
!ffmpeg -i final_output.avi final_output.mp4

In [ ]:
import os
print(os.listdir("/content"))